 **What this notebook does**
1. Samples 20 bills (5 per type: HR, S, HJRES, SJRES) from `bills.csv`.
2. Downloads each bill's detail JSON to `test_data/{bill_id}.json`.
3. Inspects the response structure: which fields are inline, which are link stubs, and whether `laws` appears.
4. Extracts each bill's sponsor (`bioguideId`, party, state) into a table.
5. Merges the sponsors with Voteview DW-NOMINATE scores on `bioguideId` + `congress`.

**Findings**
- The sponsor is a list of objects; the key is `bioguideId` (lowercase "d").
- Fields like `actions`, `cosponsors` and `committees` are only counts plus links to other endpoints.
- Voteview has duplicate rows for some members in the 118th (Schiff and Kim moved House to Senate; Manchin's two rows have different scores). Rule for now: keep the first row.


In [3]:
import pandas as pd
import json
import requests
from dotenv import load_dotenv
import time
import os
from pathlib import Path
params = {"format": "json"}
load_dotenv("../.env")
key = os.environ['CONGRESS_API_KEY']
headers = {"X-API-Key": key}

In [7]:
bills = pd.read_csv("../data/processed_bills/bills.csv")
bills.head(1)



,congress,introducedDate,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,label_latest_action_date,label_latest_action_text,label_latest_action_time,bill_id
0,118,2023-01-09,8,House,H,Proposing an amendment to the Constitution of ...,HJRES,2025-12-05,2025-12-05,https://api.congress.gov/v3/bill/118/hjres/8?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN,118HJRES8


In [8]:
# sampling some rows 

test = bills.groupby("type").sample(n=5, random_state=42)


In [19]:
for bill in test.itertuples():
    filename = f"test_data/{bill.bill_id}.json"
    url = f"https://api.congress.gov/v3/bill/{bill.congress}/{bill.type}/{bill.number}"
    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    data = response.json()
    with open(filename, "w") as f:
        json.dump(data, f)
    time.sleep(1)

    print(bill.bill_id, response.status_code)
    


118HJRES222 200
118HJRES44 200
118HJRES1 200
118HJRES117 200
118HJRES19 200
118HR4579 200
118HR7800 200
118HR7155 200
118HR8381 200
118HR853 200
118S4461 200
118S3925 200
118S4888 200
118S3142 200
118S947 200
118SJRES16 200
118SJRES23 200
118SJRES120 200
118SJRES41 200
118SJRES4 200


In [20]:
with open("test_data/118SJRES23.json", "r") as file:
    data = json.load(file)
data["bill"].keys()

dict_keys(['actions', 'committees', 'congress', 'cosponsors', 'introducedDate', 'latestAction', 'legislationUrl', 'number', 'originChamber', 'originChamberCode', 'policyArea', 'relatedBills', 'sponsors', 'subjects', 'summaries', 'textVersions', 'title', 'titles', 'type', 'updateDate', 'updateDateIncludingText'])

In [21]:
print(data["bill"].get("laws"))

None


In [22]:
print(data['bill']['sponsors'])

[{'bioguideId': 'L000571', 'firstName': 'Cynthia', 'fullName': 'Sen. Lummis, Cynthia M. [R-WY]', 'isByRequest': 'N', 'lastName': 'Lummis', 'middleName': 'M.', 'party': 'R', 'state': 'WY', 'url': 'https://api.congress.gov/v3/member/L000571?format=json'}]


# test run


In [4]:
rows = []
for file in sorted(Path("test_data").glob("*.json")):
    with open(file) as f:
        bill = json.load(f)['bill']

    sponsors = bill.get("sponsors", []) # [] if key is missing
    sponsor = sponsors[0] if sponsors else {} # {} if no sponsor

    rows.append({
        "bill_id": file.stem, # filename - .json
        "sponsor_bioguide_id": sponsor.get("bioguideId"), # none if missing
        "sponsor_party": sponsor.get("party"),
        "sponsor_state": sponsor.get("state")
    })

sponsor_df = pd.DataFrame(rows)
print(sponsor_df.shape)
print(sponsor_df["sponsor_bioguide_id"].isna().sum(), "bills with no sponsor")

(20, 4)
0 bills with no sponsor


In [9]:
# attaching sponsor to sample bills

pilot = test.merge(sponsor_df, on="bill_id", how="left", validate="one_to_one")
print(pilot.shape)

(20, 17)


In [10]:
# prepping voteview ideology becauase of dupe errors
ideology = pd.read_csv("../data/ideology_data/congress_118_ideology.csv")
ideology = ideology[ideology['chamber'].isin(["House", "Senate"])]


# rename so both tables use same key name
ideology = ideology.rename(columns={"bioguide_id": "sponsor_bioguide_id"})
ideology = ideology[["sponsor_bioguide_id", 'congress', 'nominate_dim1', 'nominate_dim2']]

# some members appear twice in one congress (Schiff, Kim moved House -> Senate; Manchin changed party)
# rule for now: keep the first row
ideology = ideology.drop_duplicates(subset=["sponsor_bioguide_id", "congress"], keep="first")
print(ideology.duplicated(subset=["sponsor_bioguide_id", "congress"]).sum(), "duplicates left")  # want 0

0 duplicates left


In [ ]:
# attaching voteview ideology

# rename so both tables use same key name
ideology = ideology.rename(columns={"bioguide_id": "sponsor_bioguide_id"})
ideology = ideology[['sponsor_bioguide_id', 'congress', 'nominate_dim1', 'nominate_dim2']]

pilot = pilot.merge(
    ideology, 
    on=['sponsor_bioguide_id', 'congress'],
    how='left',
    validate='many_to_one', # many bills per member, one member row per key
    indicator=True
)

print(pilot["_merge"].value_counts())
print(pilot[["bill_id", "sponsor_bioguide_id", "nominate_dim1", "_merge"]])


_merge
both          20
left_only      0
right_only     0
Name: count, dtype: int64
        bill_id sponsor_bioguide_id  nominate_dim1 _merge
0   118HJRES222             P000609          0.670   both
1    118HJRES44             C001116          0.827   both
2     118HJRES1             B001302          0.818   both
3   118HJRES117             A000372          0.689   both
4    118HJRES19             P000605          0.685   both
5     118HR4579             G000551         -0.598   both
6     118HR7800             C001069         -0.336   both
7     118HR7155             H001086          0.682   both
8     118HR8381             H001066         -0.311   both
9      118HR853             W000798          0.498   both
10     118S4461             D000622         -0.347   both
11     118S3925             C001070         -0.313   both
12     118S4888             W000800         -0.399   both
13     118S3142             R000595          0.616   both
14      118S947             K000393          0

: 

In [27]:
dupes = ideology[ideology["sponsor_bioguide_id"].isin(["S001150", "K000394", "M001183"])]
dupes

,sponsor_bioguide_id,congress,nominate_dim1,nominate_dim2
24,S001150,118,-0.353,-0.118
253,K000394,118,-0.428,0.268
551,M001183,118,-0.057,0.400


Duplicate entries are breaking the merge. Joe Manchin did this. 

In [28]:
ideology = ideology.drop_duplicates(subset=["sponsor_bioguide_id", "congress"], keep="first")
ideology

,sponsor_bioguide_id,congress,nominate_dim1,nominate_dim2
1,R000575,118,0.379,0.377
2,S001185,118,-0.402,0.397
3,P000609,118,0.670,0.113
4,C001054,118,0.550,0.239
5,M001212,118,0.646,-0.240
...,...,...,...,...
551,M001183,118,-0.057,0.400
553,B001230,118,-0.484,-0.123
554,J000293,118,0.648,-0.162
555,L000571,118,0.682,-0.311


In [1]:
# attaching voteview ideology
ideology = ideology[ideology['chamber'].isin(["House", "Senate"])]

# rename so both tables use same key name
ideology = ideology.rename(columns={"bioguide_id": "sponsor_bioguide_id"})
ideology = ideology[['sponsor_bioguide_id', 'congress', 'nominate_dim1', 'nominate_dim2']]

pilot = pilot.merge(
    ideology, 
    on=['sponsor_bioguide_id', 'congress'],
    how='left',
    validate='many_to_one', # many bills per member, one member row per key
    indicator=True
)

print(pilot["_merge"].value_counts())
print(pilot[["bill_id", "sponsor_bioguide_id", "nominate_dim1", "_merge"]])

NameError: name 'ideology' is not defined